# Inferencia

Recebe o caminho de uma imagem qualquer e devolve a mascara de instancias colorida
mais a contagem de objetos. Nao treina nada, so carrega o checkpoint.

Antes de rodar: `uv sync` e ter um checkpoint em `runs/`. Se ainda nao treinou nada,
roda `uv run python scripts/train.py --config configs/synthetic_baseline.yaml`, que
leva menos de 5 minutos.

In [ ]:
import sys
from pathlib import Path

# deixa o notebook rodar tanto da raiz quanto de dentro de notebooks/
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

from pa1.data.transforms import eval_transform
from pa1.models.unet import build_model
from pa1.postprocess.naive import labels_from_probability
from pa1.utils import get_device, load_checkpoint, load_config
from pa1.viz import colorize_labels, overlay_instances

device = get_device()
print('device:', device)

## Configuracao

Troca as duas variaveis abaixo pelo que voce quer rodar.

In [ ]:
CONFIG = ROOT / 'configs' / 'synthetic_baseline.yaml'
CHECKPOINT = ROOT / 'runs' / 'synthetic_baseline' / 'best.pt'

cfg = load_config(CONFIG)
model = build_model(cfg['model']).to(device)
meta = load_checkpoint(CHECKPOINT, model, map_location=device)
model.eval()
print('checkpoint da epoca', meta.get('epoch'), 'com mAP de val', round(meta.get('val_mAP', float('nan')), 4))

## A funcao

In [ ]:
@torch.no_grad()
def segmentar(path, threshold=None, min_size=None):
    """Caminho de imagem -> (labels, prob). labels e int32 com 0 no fundo."""
    pp = cfg.get('postprocess', {})
    threshold = pp.get('threshold', 0.5) if threshold is None else threshold
    min_size = pp.get('min_size', 10) if min_size is None else min_size

    image = np.asarray(Image.open(path).convert('RGB'), dtype=np.uint8)
    h, w = image.shape[:2]

    x = eval_transform()(image=image)['image']
    x = torch.from_numpy(x.transpose(2, 0, 1).copy())[None].to(device)
    prob = torch.sigmoid(model(x))[0, 0].cpu().numpy()[:h, :w]

    labels = labels_from_probability(prob, threshold=threshold, min_size=min_size)
    return labels, prob, image


def mostrar(path):
    labels, prob, image = segmentar(path)
    n = int(labels.max())

    fig, ax = plt.subplots(1, 3, figsize=(14, 4.6))
    ax[0].imshow(image); ax[0].set_title('imagem')
    ax[1].imshow(colorize_labels(labels)); ax[1].set_title(f'{n} instancias')
    ax[2].imshow(overlay_instances(image / 255.0, labels)); ax[2].set_title('overlay')
    for a in ax:
        a.axis('off')
    plt.tight_layout(); plt.show()
    return n

## Rodando

Se nao tiver imagem a mao, a celula abaixo escreve uma do dataset sintetico em disco
e usa ela, so pra dar pra rodar o notebook do zero.

In [ ]:
from pa1.data.synthetic import SyntheticEllipsesDataset

IMAGEM = None  # coloca aqui o caminho de uma imagem sua

if IMAGEM is None:
    img, _ = SyntheticEllipsesDataset(length=1, seed=99).raw(0)
    IMAGEM = ROOT / 'runs' / 'exemplo.png'
    IMAGEM.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray((img * 255).astype(np.uint8)).save(IMAGEM)

n = mostrar(IMAGEM)
print('contagem:', n)

Lembrando que enquanto a Parte 2 nao entra, isso aqui usa o decoder ingenuo
(limiar + componentes conexos), entao objeto encostado sai grudado num blob so.
Quando o watershed da trilha A ficar pronto, e so trocar a chamada de
`labels_from_probability` por `labels_from_boundary_head`.